# 7.1 RLHF（基于人类反馈的强化学习）

RLHF是当前大模型对齐的主流方法，通过训练奖励模型来近似人类偏好，再用PPO优化策略模型。它是ChatGPT等产品的核心技术之一。

## 为什么需要RLHF？

SFT（监督微调）后的模型可能：
- 生成有害、偏见或不准确的内容
- 无法区分好回复和差回复
- 倾向于生成冗长但低质量的回答

RLHF通过人类偏好信号引导模型朝期望方向优化。

## RLHF三步流程

1. **SFT阶段**：在高质量数据上监督微调，获得基线策略模型
2. **RM阶段**：收集人类偏好数据，训练奖励模型学习人类偏好排序
3. **PPO阶段**：用强化学习优化策略模型最大化奖励，同时KL惩罚防止偏离

## 核心公式

**奖励模型（Bradley-Terry模型）**：
$$L_{RM} = -\mathbb{E}[\log \sigma(r(x, y_w) - r(x, y_l))]$$

**PPO目标**：
$$\max_\theta \mathbb{E}_{x \sim D, y \sim \pi_\theta}[r(x,y)] - \beta \cdot D_{KL}(\pi_\theta || \pi_{ref})$$

**PPO-Clip**：
$$L^{CLIP} = \min(ratio \cdot A, \text{clip}(ratio, 1-\varepsilon, 1+\varepsilon) \cdot A)$$
其中 $ratio = \pi_\theta(a|s) / \pi_{old}(a|s)$

## 1. 奖励模型训练

奖励模型是RLHF的核心组件，它学习人类偏好排序，为PPO阶段提供奖励信号。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy

torch.manual_seed(42)

class RewardModel(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_model, 256), nn.ReLU(),
            nn.LayerNorm(256),
            nn.Linear(256, 128), nn.ReLU(),
            nn.LayerNorm(128),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.encoder(x).squeeze(-1)

def bradley_terry_loss(reward_model, x, y_chosen, y_rejected, margin=0.0):
    r_chosen = reward_model(torch.cat([x, y_chosen], dim=-1) if x.shape != y_chosen.shape else y_chosen)
    r_rejected = reward_model(torch.cat([x, y_rejected], dim=-1) if x.shape != y_rejected.shape else y_rejected)
    loss = -F.logsigmoid(r_chosen - r_rejected - margin).mean()
    with torch.no_grad():
        accuracy = (r_chosen > r_rejected).float().mean()
    return loss, accuracy

reward_model = RewardModel(d_model=128)
optimizer = torch.optim.AdamW(reward_model.parameters(), lr=1e-3, weight_decay=0.01)

d_model = 128
n_pairs = 32

x_prompt = torch.randn(n_pairs, d_model)
y_chosen = x_prompt + 0.3 * torch.randn(n_pairs, d_model)
y_rejected = x_prompt + 0.8 * torch.randn(n_pairs, d_model)

print('=== Reward Model Training ===')
for epoch in range(50):
    loss, acc = bradley_terry_loss(reward_model, x_prompt, y_chosen, y_rejected, margin=0.1)
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}: Loss={loss.item():.4f}, Accuracy={acc:.2%}')

r_c = reward_model(y_chosen)
r_r = reward_model(y_rejected)
print(f'\nFinal - Chosen rewards: mean={r_c.mean():.3f}, std={r_c.std():.3f}')
print(f'Final - Rejected rewards: mean={r_r.mean():.3f}, std={r_r.std():.3f}')
print(f'Final preference accuracy: {(r_c > r_r).float().mean():.2%}')

## 2. 奖励模型集成与不确定性估计

工业实践中，单个奖励模型容易出现reward hacking（策略模型利用奖励模型的漏洞获取高分）。使用奖励模型集成可以提高鲁棒性。

In [ ]:
class RewardEnsemble:
    def __init__(self, n_models=5, d_model=128):
        self.models = [RewardModel(d_model) for _ in range(n_models)]
        self.n_models = n_models

    def predict(self, x, reduction='mean'):
        rewards = torch.stack([m(x).detach() for m in self.models])
        if reduction == 'mean':
            return rewards.mean(dim=0)
        elif reduction == 'min':
            return rewards.min(dim=0).values
        elif reduction == 'vote':
            return rewards
        return rewards.mean(dim=0)

    def uncertainty(self, x):
        rewards = torch.stack([m(x).detach() for m in self.models])
        return rewards.std(dim=0)

    def train_models(self, x, y_chosen, y_rejected, n_epochs=30, lr=1e-3):
        for i, model in enumerate(self.models):
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
            for epoch in range(n_epochs):
                loss, _ = bradley_terry_loss(model, x, y_chosen, y_rejected)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            print(f'  Model {i+1}/{self.n_models} trained')

ensemble = RewardEnsemble(n_models=5, d_model=128)
print('Training ensemble of reward models...')
ensemble.train_models(x_prompt, y_chosen, y_rejected, n_epochs=30)

test_x = torch.randn(8, d_model)
test_y = test_x + 0.3 * torch.randn(8, d_model)
mean_reward = ensemble.predict(test_y, reduction='mean')
min_reward = ensemble.predict(test_y, reduction='min')
uncertainty = ensemble.uncertainty(test_y)

print(f'\n=== Ensemble Prediction ===')
print(f'Mean rewards: {mean_reward.tolist()}')
print(f'Min rewards:  {min_reward.tolist()}')
print(f'Uncertainty:  {uncertainty.tolist()}')
print(f'\nKey: Min reduction is conservative, reduces reward hacking.')
print(f'High uncertainty signals unreliable reward estimates.')

## 3. PPO优化阶段

PPO（Proximal Policy Optimization）是RLHF的核心优化算法，通过clip机制限制策略更新幅度，保证训练稳定性。

### PPO关键组件
- **策略模型（Actor）**：生成文本的策略
- **价值模型（Critic）**：估计状态价值，计算优势函数
- **参考模型（Reference）**：KL惩罚的参考点
- **奖励模型**：提供奖励信号

### GAE（广义优势估计）
$$A^{GAE}(t) = \sum_{l=0}^{T-t}(\gamma\lambda)^l \delta_{t+l}$$
其中 $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$

In [ ]:
class PolicyModel(nn.Module):
    def __init__(self, d_model=128, vocab_size=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 256), nn.ReLU(),
            nn.LayerNorm(256),
            nn.Linear(256, vocab_size)
        )
    def forward(self, x):
        return self.net(x)

class ValueModel(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 256), nn.ReLU(),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    advantages = []
    gae = 0
    next_value = 0
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * next_value - values[t]
        gae = delta + gamma * lam * gae
        advantages.insert(0, gae)
        next_value = values[t]
    advantages = torch.tensor(advantages)
    returns = advantages + values
    return advantages, returns

def ppo_step(policy, ref_policy, value_model, reward_model, optimizer_policy,
             optimizer_value, x, epsilon=0.2, beta_kl=0.1, value_coef=0.5,
             entropy_coef=0.01):
    batch_size = x.shape[0]
    vocab_size = policy.net[-1].out_features

    with torch.no_grad():
        old_logits = policy(x)
        old_log_probs = F.log_softmax(old_logits, dim=-1)
        actions = old_log_probs.exp().multinomial(1).squeeze(-1)
        old_action_log_probs = old_log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)
        action_emb = F.one_hot(actions, vocab_size).float()
        rewards = reward_model(action_emb)
        values = value_model(x)
        ref_logits = ref_policy(x)
        ref_log_probs = F.log_softmax(ref_logits, dim=-1)
        ref_action_log_probs = ref_log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)

    advantages, returns = compute_gae(rewards, values)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    n_ppo_epochs = 4
    total_policy_loss = 0
    total_value_loss = 0
    total_kl = 0

    for _ in range(n_ppo_epochs):
        new_logits = policy(x)
        new_log_probs = F.log_softmax(new_logits, dim=-1)
        new_action_log_probs = new_log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)

        ratio = (new_action_log_probs - old_action_log_probs).exp()
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
        policy_loss = -torch.min(surr1, surr2).mean()

        with torch.no_grad():
            kl_div = (new_action_log_probs - ref_action_log_probs).mean()
        kl_penalty = beta_kl * kl_div

        entropy = -(new_log_probs.exp() * new_log_probs).sum(dim=-1).mean()
        entropy_bonus = -entropy_coef * entropy

        total_policy = policy_loss + kl_penalty + entropy_bonus
        optimizer_policy.zero_grad()
        total_policy.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
        optimizer_policy.step()

        new_values = value_model(x)
        value_loss = F.mse_loss(new_values, returns)
        optimizer_value.zero_grad()
        value_coef * value_loss.backward()
        optimizer_value.step()

        total_policy_loss += policy_loss.item()
        total_value_loss += value_loss.item()
        total_kl += kl_div.item()

    metrics = {
        'policy_loss': total_policy_loss / n_ppo_epochs,
        'value_loss': total_value_loss / n_ppo_epochs,
        'kl_divergence': total_kl / n_ppo_epochs,
        'mean_reward': rewards.mean().item(),
        'approx_kl': ((ratio - 1) - ratio.log()).mean().item(),
    }
    return metrics

policy = PolicyModel(d_model=128, vocab_size=100)
ref_policy = PolicyModel(d_model=128, vocab_size=100)
ref_policy.load_state_dict(policy.state_dict())
for p in ref_policy.parameters():
    p.requires_grad = False
value_model = ValueModel(d_model=128)

optimizer_policy = torch.optim.AdamW(policy.parameters(), lr=1e-4)
optimizer_value = torch.optim.AdamW(value_model.parameters(), lr=1e-3)

x = torch.randn(32, 128)

print('=== PPO Training Loop ===')
for step in range(20):
    metrics = ppo_step(policy, ref_policy, value_model, reward_model,
                       optimizer_policy, optimizer_value, x)
    if (step + 1) % 5 == 0:
        print(f'Step {step+1:3d}: reward={metrics["mean_reward"]:.3f}, '
              f'kl={metrics["kl_divergence"]:.4f}, '
              f'p_loss={metrics["policy_loss"]:.4f}, '
              f'v_loss={metrics["value_loss"]:.4f}')

print(f'\nKey: PPO balances reward maximization with KL penalty.')
print(f'KL divergence should stay small (< 0.1) for stable training.')

## 4. KL散度控制策略

KL惩罚是RLHF训练稳定性的关键。工业实践中使用多种策略来控制KL散度：

1. **固定系数KL惩罚**：$L = L_{PPO} + \beta \cdot D_{KL}$
2. **自适应KL控制器**：根据目标KL值动态调整$\beta$
3. **KL散度早停**：当KL超过阈值时停止更新
4. **KL退火**：训练过程中逐步减小$\beta$

In [ ]:
class AdaptiveKLController:
    def __init__(self, init_kl_coef=0.1, target_kl=0.1, kl_horizon=10000):
        self.kl_coef = init_kl_coef
        self.target_kl = target_kl
        self.kl_horizon = kl_horizon

    def update(self, current_kl, step):
        ratio = current_kl / (self.target_kl + 1e-8)
        if ratio > 1.5:
            self.kl_coef *= 2.0
        elif ratio < 0.67:
            self.kl_coef *= 0.5
        self.kl_coef = max(1e-6, min(self.kl_coef, 10.0))
        return self.kl_coef

class KLEarlyStopper:
    def __init__(self, max_kl=0.2, patience=3):
        self.max_kl = max_kl
        self.patience = patience
        self.violations = 0

    def should_stop(self, current_kl):
        if current_kl > self.max_kl:
            self.violations += 1
        else:
            self.violations = max(0, self.violations - 1)
        return self.violations >= self.patience

class KLAnnealer:
    def __init__(self, init_coef=0.2, final_coef=0.01, total_steps=1000):
        self.init_coef = init_coef
        self.final_coef = final_coef
        self.total_steps = total_steps

    def get_coef(self, step):
        progress = min(step / self.total_steps, 1.0)
        return self.init_coef * (1 - progress) + self.final_coef * progress

kl_adaptive = AdaptiveKLController(init_kl_coef=0.1, target_kl=0.1)
kl_stopper = KLEarlyStopper(max_kl=0.2)
kl_annealer = KLAnnealer(init_coef=0.2, final_coef=0.01, total_steps=100)

simulated_kls = [0.05, 0.08, 0.15, 0.25, 0.30, 0.12, 0.06, 0.03]

print('=== KL Divergence Control Strategies ===')
print(f'\n--- Adaptive KL Controller ---')
for i, kl in enumerate(simulated_kls):
    coef = kl_adaptive.update(kl, i)
    print(f'  Step {i}: KL={kl:.3f}, target={kl_adaptive.target_kl:.3f}, '
          f'adjusted_coef={coef:.4f}')

print(f'\n--- KL Early Stopper ---')
for i, kl in enumerate(simulated_kls):
    stop = kl_stopper.should_stop(kl)
    print(f'  Step {i}: KL={kl:.3f}, violations={kl_stopper.violations}, '
          f'should_stop={stop}')

print(f'\n--- KL Annealer ---')
for step in [0, 25, 50, 75, 100]:
    coef = kl_annealer.get_coef(step)
    print(f'  Step {step:3d}: KL coefficient={coef:.4f}')

print(f'\nKey: Adaptive controller is most commonly used in production.')
print(f'Early stopping prevents catastrophic divergence.')
print(f'Annealing allows more exploration early, more stability later.')

## 5. Reward Hacking检测与防御

Reward hacking是RLHF中最常见的问题：策略模型学会了利用奖励模型的漏洞获取高分，而不是真正提升输出质量。

### 常见Reward Hacking模式
- **长度 hacking**：生成长回复获取更高奖励
- **格式 hacking**：学习特定格式获取高分
- **重复 hacking**：重复高分片段
- **关键词 hacking**：堆砌奖励模型偏好的关键词

In [ ]:
class RewardHackingDetector:
    def __init__(self, window_size=10, threshold=0.3):
        self.window_size = window_size
        self.threshold = threshold
        self.reward_history = []
        self.human_eval_history = []

    def check_length_hacking(self, responses, rewards, max_length_ratio=2.0):
        lengths = torch.tensor([len(r) if isinstance(r, (list, str)) else r.shape[0]
                               for r in responses], dtype=torch.float)
        mean_len = lengths.mean()
        long_mask = lengths > mean_len * max_length_ratio
        if long_mask.any():
            long_rewards = rewards[long_mask].mean()
            short_rewards = rewards[~long_mask].mean()
            return long_rewards - short_rewards > self.threshold
        return False

    def check_reward_human_gap(self, model_reward, human_score):
        self.reward_history.append(model_reward)
        self.human_eval_history.append(human_score)
        if len(self.reward_history) >= self.window_size:
            recent_rewards = self.reward_history[-self.window_size:]
            recent_human = self.human_eval_history[-self.window_size:]
            reward_trend = recent_rewards[-1] - recent_rewards[0]
            human_trend = recent_human[-1] - recent_human[0]
            return reward_trend > 0 and human_trend < 0
        return False

    def check_repetition(self, response, max_repeat=3):
        if isinstance(response, torch.Tensor):
            tokens = response.tolist()
        else:
            tokens = response
        for n in range(1, len(tokens) // 2 + 1):
            for i in range(len(tokens) - n * max_repeat):
                pattern = tokens[i:i+n]
                count = 0
                j = i
                while j + n <= len(tokens) and tokens[j:j+n] == pattern:
                    count += 1
                    j += n
                if count >= max_repeat:
                    return True
        return False

detector = RewardHackingDetector(window_size=5)

simulated_responses = [torch.randn(torch.randint(10, 50, (1,)).item()) for _ in range(8)]
simulated_rewards = torch.tensor([0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.97, 0.98])

print('=== Reward Hacking Detection ===')
print(f'\nLength hacking check:')
is_hacking = detector.check_length_hacking(simulated_responses, simulated_rewards)
print(f'  Detected: {is_hacking}')

print(f'\nReward-Human gap check:')
model_rewards = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.97, 0.98]
human_scores =  [0.5, 0.55, 0.6, 0.58, 0.55, 0.50, 0.48, 0.45]
for mr, hs in zip(model_rewards, human_scores):
    gap = detector.check_reward_human_gap(mr, hs)
    if gap:
        print(f'  WARNING: Reward increasing but human score decreasing!')
        break
else:
    print(f'  No reward-human gap detected in this window')

print(f'\nRepetition check:')
normal_response = [1, 2, 3, 4, 5, 6, 7, 8]
repeat_response = [1, 2, 1, 2, 1, 2, 1, 2, 1, 2]
print(f'  Normal response has repetition: {detector.check_repetition(normal_response)}')
print(f'  Repeated response has repetition: {detector.check_repetition(repeat_response)}')

print(f'\nKey: Monitor reward-human gap as the primary hacking indicator.')
print(f'Periodic human evaluation is essential for production RLHF.')

## 6. 完整RLHF训练Pipeline

工业级RLHF训练需要考虑数据管理、训练调度、评估监控等多个方面。

In [ ]:
class RLHFTrainer:
    def __init__(self, policy, ref_policy, reward_model, value_model,
                 kl_coef=0.1, target_kl=0.1, gamma=0.99, lam=0.95,
                 ppo_epochs=4, epsilon=0.2, value_coef=0.5, entropy_coef=0.01,
                 max_grad_norm=0.5):
        self.policy = policy
        self.ref_policy = ref_policy
        self.reward_model = reward_model
        self.value_model = value_model
        self.kl_coef = kl_coef
        self.target_kl = target_kl
        self.gamma = gamma
        self.lam = lam
        self.ppo_epochs = ppo_epochs
        self.epsilon = epsilon
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm

        self.optimizer_policy = torch.optim.AdamW(
            policy.parameters(), lr=1e-5, weight_decay=0.01,
            betas=(0.9, 0.95))
        self.optimizer_value = torch.optim.AdamW(
            value_model.parameters(), lr=1e-4)
        self.scheduler_policy = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer_policy, T_max=1000, eta_min=1e-6)
        self.kl_controller = AdaptiveKLController(init_kl_coef=kl_coef, target_kl=target_kl)
        self.hacking_detector = RewardHackingDetector()
        self.metrics_history = []

    @torch.no_grad()
    def generate_rollout(self, x):
        vocab_size = self.policy.net[-1].out_features
        old_logits = self.policy(x)
        old_log_probs = F.log_softmax(old_logits, dim=-1)
        actions = old_log_probs.exp().multinomial(1).squeeze(-1)
        old_action_log_probs = old_log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)
        action_emb = F.one_hot(actions, vocab_size).float()
        rewards = self.reward_model(action_emb)
        values = self.value_model(x)
        ref_logits = self.ref_policy(x)
        ref_log_probs = F.log_softmax(ref_logits, dim=-1)
        ref_action_log_probs = ref_log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)
        return actions, old_action_log_probs, ref_action_log_probs, rewards, values

    def train_step(self, x):
        actions, old_log_probs, ref_log_probs, rewards, values = self.generate_rollout(x)
        advantages, returns = compute_gae(rewards, values, self.gamma, self.lam)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        step_metrics = {}
        for _ in range(self.ppo_epochs):
            new_logits = self.policy(x)
            new_log_probs = F.log_softmax(new_logits, dim=-1)
            new_action_log_probs = new_log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)

            ratio = (new_action_log_probs - old_log_probs).exp()
            surr1 = ratio * advantages
            surr2 = torch.clamp(ratio, 1 - self.epsilon, 1 + self.epsilon) * advantages
            policy_loss = -torch.min(surr1, surr2).mean()

            kl_div = (new_action_log_probs - ref_log_probs).mean()
            kl_penalty = self.kl_controller.kl_coef * kl_div

            entropy = -(new_log_probs.exp() * new_log_probs).sum(dim=-1).mean()
            total_policy_loss = policy_loss + kl_penalty - self.entropy_coef * entropy

            self.optimizer_policy.zero_grad()
            total_policy_loss.backward()
            nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
            self.optimizer_policy.step()

            new_values = self.value_model(x)
            value_loss = F.mse_loss(new_values, returns)
            self.optimizer_value.zero_grad()
            self.value_coef * value_loss.backward()
            self.optimizer_value.step()

        self.kl_controller.update(kl_div.item(), len(self.metrics_history))
        self.scheduler_policy.step()

        step_metrics = {
            'mean_reward': rewards.mean().item(),
            'policy_loss': policy_loss.item(),
            'value_loss': value_loss.item(),
            'kl_divergence': kl_div.item(),
            'kl_coef': self.kl_controller.kl_coef,
            'entropy': entropy.item(),
            'lr': self.optimizer_policy.param_groups[0]['lr'],
        }
        self.metrics_history.append(step_metrics)
        return step_metrics

policy = PolicyModel(d_model=128, vocab_size=100)
ref_policy = PolicyModel(d_model=128, vocab_size=100)
ref_policy.load_state_dict(policy.state_dict())
for p in ref_policy.parameters():
    p.requires_grad = False
value_model = ValueModel(d_model=128)
rm = RewardModel(d_model=128)

trainer = RLHFTrainer(policy, ref_policy, rm, value_model)

x_train = torch.randn(32, 128)
print('=== Full RLHF Training Pipeline ===')
for step in range(30):
    metrics = trainer.train_step(x_train)
    if (step + 1) % 10 == 0:
        print(f'Step {step+1:3d}: reward={metrics["mean_reward"]:.3f}, '
              f'kl={metrics["kl_divergence"]:.4f}, '
              f'kl_coef={metrics["kl_coef"]:.4f}, '
              f'entropy={metrics["entropy"]:.3f}, '
              f'lr={metrics["lr"]:.2e}')

print(f'\nKey: Full pipeline includes adaptive KL, LR scheduling, gradient clipping.')
print(f'Monitor KL divergence and reward-human gap for training health.')

## 7. RLHF工业实践要点

### 数据质量
- **偏好数据质量** > 数量，高质量标注者的数据价值远高于众包
- **标注者一致性**：测量标注者间一致性（Cohen's Kappa），过滤低质量标注
- **数据多样性**：覆盖不同任务类型、难度级别和安全边界

### 训练技巧
- **RM训练**：使用margin-based Bradley-Terry loss，增加0.1-0.5的margin
- **PPO超参**：epsilon=0.2, kl_coef=0.05-0.2, value_coef=0.5, entropy_coef=0.01
- **批量大小**：PPO需要较大batch（512-4096），小batch导致高方差
- **学习率**：策略模型用1e-6到5e-6，价值模型用1e-5到5e-5

### 常见问题与解决方案
| 问题 | 症状 | 解决方案 |
|------|------|----------|
| Reward hacking | 奖励上升但质量下降 | RM集成、定期人工评估 |
| KL爆炸 | 策略偏离参考太远 | 自适应KL控制器、早停 |
| 价值函数不准 | 优势估计偏差大 | 增大value_coef、更多训练 |
| 模式坍塌 | 输出多样性降低 | 增大entropy_coef、降低kl_coef |
| 训练不稳定 | loss震荡、NaN | 减小学习率、增大batch |

### 与DPO的选择
- **选RLHF**：需要在线探索、有充足计算资源、追求最优性能
- **选DPO**：计算资源有限、偏好数据充足、追求训练简洁性
- **混合方案**：先用DPO做初步对齐，再用RLHF精调